In [1]:
%pip install python-dotenv --upgrade --quiet langchain langchain-google-genai


**Part 1a: LangChain Setup & Models**

In [2]:
from dotenv import load_dotenv
load_dotenv()
import getpass
import os
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")



Enter your Google API Key: ··········


In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Model A: The "Accountant" (Precision)
llm_focused = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)

# Model B: The "Poet" (Creativity)
llm_creative = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=1.0)

In [4]:
prompt = "Describe the seventh wonder of India ,the Taj Mahal"

print("--- FOCUSED (Temp=0) ---")
print(f"Run 1: {llm_focused.invoke(prompt).content}")
print(f"Run 2: {llm_focused.invoke(prompt).content}")

--- FOCUSED (Temp=0) ---
Run 1: The Taj Mahal, often hailed as the "crown jewel of Muslim art in India" and a universally recognized symbol of India itself, is not just a monument but a poem in marble. While there isn't an official list of "seven wonders of India," the Taj Mahal undoubtedly holds a paramount position as one of the world's most iconic and breathtaking architectural marvels.

Here's a description of this magnificent structure:

1.  **A Monument to Love:**
    At its heart, the Taj Mahal is a profound testament to eternal love. It was commissioned in 1632 by the Mughal emperor Shah Jahan to house the tomb of his beloved wife, Mumtaz Mahal, who died giving birth to their 14th child. It is widely regarded as the greatest architectural tribute to love ever built.

2.  **Location:**
    It stands majestically on the southern bank of the Yamuna River in the city of Agra, Uttar Pradesh, India.

3.  **Architectural Grandeur and Material:**
    *   **Pristine White Marble:** The 

In [6]:
print(type(llm_focused))
print(type(llm_creative))

<class 'langchain_google_genai.chat_models.ChatGoogleGenerativeAI'>
<class 'langchain_google_genai.chat_models.ChatGoogleGenerativeAI'>


** Part 1b: Prompts & Parsers**

In [7]:
from langchain_core.messages import SystemMessage, HumanMessage



In [8]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="You are a knowledgeable historian."),
    HumanMessage(content="Describe the Taj Mahal.")
]

response = llm_focused.invoke(messages)
print(response.content)


The Taj Mahal is an iconic mausoleum located in Agra, Uttar Pradesh, India. It was commissioned in 1632 by the Mughal emperor Shah Jahan to house the tomb of his beloved wife, Mumtaz Mahal. It is widely regarded as one of the most beautiful buildings in the world and a universal symbol of eternal love.

Here's a detailed description:

1.  **Purpose and Origin:** At its heart, the Taj Mahal is a tomb, a grand final resting place for Mumtaz Mahal, who died giving birth to their 14th child. Shah Jahan, consumed by grief, vowed to build a monument worthy of her beauty and their love, a promise that resulted in this unparalleled architectural masterpiece.

2.  **Architecture and Design:**
    *   **Material:** The most striking feature is its construction from **shimmering white marble**, quarried from Makrana, Rajasthan. This marble appears to change color with the shifting light of the sun and moon, from pearly white to golden, pink, and even grey.
    *   **Symmetry:** The entire complex

In [9]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ("system", "You are a translator. Translate {input_language} to {output_language}."),
    ("human", "{text}")
])

# We can check what inputs it expects
print(f"Required variables: {template.input_variables}")

Required variables: ['input_language', 'output_language', 'text']


In [11]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

# Raw Message
raw_msg = llm_focused.invoke("Hi")
print(f"Raw Type: {type(raw_msg)}")

# Parsed String
clean_text = parser.invoke(raw_msg)
print(f"Parsed Type: {type(clean_text)}")
print(f"Content: {clean_text}")

Raw Type: <class 'langchain_core.messages.ai.AIMessage'>
Parsed Type: <class 'langchain_core.messages.base.TextAccessor'>
Content: Hi there! How can I help you today?


**1c.LCEL**

In [12]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
template = ChatPromptTemplate.from_template("Tell me a fun fact about {topic}.")
parser = StrOutputParser()

In [13]:
prompt_value = template.invoke({"topic": "Cats"})
response_obj = llm.invoke(prompt_value)
final_text = parser.invoke(response_obj)

print(final_text)

Here's a fun one:

Cats can make over **100 different sounds**, whereas dogs can only make about 10! This includes everything from purrs and meows to chirps, trills, hisses, and growls, each often with a specific meaning.


In [14]:
chain = template | llm | parser
print(chain.invoke({"topic": "Fishes"}))

Here's a fun one:

Did you know that most fish don't have eyelids? This means they can't close their eyes! So, when a fish "sleeps" or rests, it does so with its eyes wide open, always looking out!


**ASSIGNMENT**

In [21]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

template = ChatPromptTemplate.from_template(
    "What year was the movie {movie} released? Also calculate how many years ago that was from 2025."
)

chain = template | llm_focused | StrOutputParser()

print(chain.invoke({"movie": "Willy Wonka"}))


The movie *Willy Wonka & the Chocolate Factory* (starring Gene Wilder) was released in **1971**.

From 2025, that was **54 years ago** (2025 - 1971 = 54).
